# Fine-tune Llama 2 with QLoRA

Structured, GitHub-ready version of the original Colab notebook. This notebook is a thin wrapper around `scripts/train.py` and `scripts/evaluate.py` so the same code runs here, from the CLI (`fine_tune_llama2.py`), or when imported.

**Compatibility note:** the original notebook pinned 2023-era package versions (`transformers==4.31.0`, `trl==0.4.7`, `peft==0.4.0`, `accelerate==0.21.0`) that only have wheels for Python ≤ 3.10. This project uses modern floor versions from `requirements.txt` that ship Python 3.13 wheels, and the training code targets the current `trl` `SFTConfig`/`SFTTrainer` API (see `scripts/train.py`).

## Step 1: Install requirements

In [ ]:
%pip install -q -r requirements.txt

## Step 2: Imports

In [ ]:
import sys
sys.path.append('..')  # if running from notebooks/ subfolder; no-op otherwise

from scripts.train import load_config, train, merge_and_save, push_to_hub
from scripts.evaluate import run_generation

## Llama 2 chat prompt template

```
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_prompt} [/INST] {model_answer} </s>
```

- System prompt is optional (guides the model)
- User prompt is required (the instruction)
- Model answer is required

Dataset used here (`mlabonne/guanaco-llama2-1k`) is already reformatted to this template -- see `data/dataset_info.json`. No prompt template is needed for the base (non-chat) Llama 2 model.

## Step 3: Load configuration

All hyperparameters (QLoRA rank/alpha/dropout, 4-bit quantization settings, `TrainingArguments`, SFT settings) live in `config.yaml` instead of notebook variables, so the same config drives the notebook, the CLI, and the scripts.

In [ ]:
cfg = load_config("../config.yaml")
cfg

## Step 4: Fine-tune (QLoRA, 4-bit)

Loads the dataset, quantizes the base model to 4-bit (NF4), attaches a LoRA adapter, and runs `SFTTrainer.train()`. Equivalent to Steps 3-4 of the original notebook, wrapped in `scripts/train.py::train()`.

In [ ]:
adapter_path = train(cfg)
print(f'Adapter saved to: {adapter_path}')

## Step 5: Check training curves on TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/logs

## Step 6: Quick generation check

Runs the fine-tuned adapter through the Llama 2 prompt template on a test prompt.

In [ ]:
run_generation(adapter_path, [cfg["inference"]["default_prompt"]], cfg["inference"]["max_length"])

## Step 7: Merge LoRA weights into the base model

Reloads the base model in FP16 and merges the LoRA adapter (`peft.merge_and_unload`), since the adapter alone isn't a directly loadable standalone model.

In [ ]:
merged_dir = merge_and_save(cfg, adapter_path)
print(f"Merged model saved to: {merged_dir}")

## Step 8: Push to the Hugging Face Hub (optional)

Set `HF_TOKEN` in your environment (or a `.env` file) before running this cell -- no interactive `huggingface-cli login` prompt required.

In [ ]:
# push_to_hub(cfg, merged_dir, "your-username/Llama-2-7b-chat-finetune")